# 06 — Main Grid Operations Agent
Loads the tools from notebook 05, initializes the Databricks-hosted LLM, creates the LangGraph ReAct agent, and runs end-to-end smoke tests.

Run notebooks **01 → 05** successfully before running this notebook.

## Load agent tools

In [ ]:
%run ./05_agent_tools

In [ ]:
from databricks_langchain import ChatDatabricks
from langgraph.prebuilt import create_react_agent

LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

llm = ChatDatabricks(
    model=LLM_MODEL,
    temperature=0,
)

## System instructions

In [ ]:
SYSTEM_PROMPT = """
You are a PGCB Grid Operations Decision-Support Agent.

Rules:
1. Never invent electricity demand, generation, import, load-shedding, or forecast values.
2. Use get_grid_data for historical grid information.
3. When tomorrow's demand is required, call forecast_next_day_demand.
4. forecast_next_day_demand returns a numeric MW value. Pass that numeric value directly
   as predicted_demand_mw to calculate_supply_gap.
5. Use calculate_supply_gap whenever forecast demand must be compared with expected
   generation and imports.
6. Use recommend_grid_action only after obtaining a supply gap and reserve margin.
7. Clearly distinguish model forecasts, historical values, and user-provided assumptions.
8. The risk thresholds are project-defined and are not official PGCB operating policy.
9. You provide decision support only and do not control real grid infrastructure.
"""

## Create agent

In [ ]:
agent = create_react_agent(
    model=llm,
    tools=tools,
)

## Smoke test 1 — calculation and recommendation

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "If forecast demand is 14500 MW, generation is 12500 MW "
                "and imports are 1500 MW, what is the grid risk?"
            ),
        },
    ]
})
print(result["messages"][-1].content)

## Smoke test 2 — model forecast → gap calculation → recommendation

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "Will there be enough electricity supply tomorrow if expected generation "
                "is 12500 MW and expected imports are 1500 MW?"
            ),
        },
    ]
})
print(result["messages"][-1].content)

## Smoke test 3 — historical retrieval

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Summarize grid conditions from 2025-06-01 through 2025-06-07.",
        },
    ]
})
print(result["messages"][-1].content)